# Lektion 18: Sikring af AI-agenter med kryptografiske kvitteringer

## Praktisk notesbog

Denne notesbog gennemgår fire øvelser:

1. **Signér din første kvittering** for et agentværktøjs kald og verificer den.
2. **Manipuler kvitteringen** og se verifikationen fejle.
3. **Byg en kæde af tre kvitteringer** og bekræft kædens integritet.
4. **Indpak et Microsoft Agent Framework-værktøjskald** således at hver handling udsender en kvittering.

Alle kryptografiske primitive importeres fra veldrevne biblioteker (`pynacl` for Ed25519, `jcs` for RFC 8785 kanonisk JSON, `hashlib` fra Python standardbiblioteket for SHA-256). Kvitteringslogikken selv er almindelig Python, som du kan læse og ændre.

Kør cellerne i rækkefølge. Hvert afsnit er kort og selvstændigt.


## Installation

Installer de to afhængigheder. Begge har tilladende licenser (Apache-2.0 / MIT).


In [1]:
%pip install -q pynacl jcs

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import CryptoError
from jcs import canonicalize

## Hjælpeværktøjer

Disse to hjælpere håndterer base64url-kodning (uden udfyldning) og SHA-256-hashning af vilkårlige objekter. De holder resten af notebogen fokuseret på kvitteringslogikken selv.


In [3]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """
    Decode unpadded base64url strictly, raising ValueError for anything else.

    Python's decoder skips characters outside the alphabet and also accepts
    padding, the standard +/ alphabet and non-zero spare bits, so the same key
    or signature could be written several ways. Re-encoding and comparing
    accepts only the one form that b64url_nopad produces.
    """
    padding = "=" * ((4 - len(s) % 4) % 4)
    data = base64.urlsafe_b64decode(s + padding)
    if b64url_nopad(data) != s:
        raise ValueError("not canonical unpadded base64url")
    return data

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Afsnit 1: Underskriv din første kvittering

Forestil dig, at vores agent for **Contoso Travel** lige har fundet flyrejser fra Sydney til Los Angeles for en kunde. Vi vil registrere gatewayens erklæring om dette værktøjsopkald, så en fremtidig revisor kan verificere det under en separat fastgjort udstedernøgle.

### Trin 1.1: Generer en underskriftsnøgle

I produktionen ville agentens underskriftsnøgle være placeret i en hardware sikkerhedsmodul (HSM), Azure Key Vault eller en lignende beskyttet lager. Til denne lektion genererer vi en frisk nøgle i hukommelsen.


In [4]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
# Trust configuration is out of band. Never trust a receipt only because it embeds a key.
TRUSTED_PUBLIC_KEYS = {public_key_b64}
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

Public key (Ed25519, 32 bytes): NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM


### Trin 1.2: Byg kvitteringsdata

Payloaden indeholder alt, hvad vi ønsker, at kvitteringen skal bevidne: hvem der handlede, hvilket værktøj, med hvilke argumenter, hvad der kom tilbage, under hvilken politik, og hvornår. Vi hasher argumenterne og resultatet i stedet for at inkludere dem inline, så kvitteringen ikke lækker følsomt indhold.


In [5]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null
}


### Trin 1.3: Underskriv og saml kvitteringen

Tre trin:

1. Kanoniser nyttelasten ved brug af JCS, så to implementeringer, der producerer den samme logiske kvittering, producerer byte-identiske bytes.
2. Underskriv de kanoniske bytes direkte med Ed25519 privatnøglen. PureEdDSA hasher meddelelsen internt, så en ekstra forhånds-hash ville ændre protokollen.

Signaturen vedhæftes derefter til den oprindelige nyttelast for at producere den endelige kvittering.


In [6]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    signature_bytes = signing_key.sign(canonical).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null,
  "signature": {
    "alg": "EdDSA",
    "sig": "NnH0T6lpjVThg1RS29rEu3OG1FptWbQgX18Yt1JAPjNnTyje2Luozt5_0gcdxddQXoS6eH6Vlg5MTcKLrvWnBQ",
    "public_key": "NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM"
  }
}


### Trin 1.4: Bekræft kvitteringen

Bekræftelse omvender processen. Vi fjerner signaturen, genberegner de kanoniske bytes og kontrollerer signaturen mod udstederens offentlige nøgle.

Bekræftelse foregår offline: ingen service at kalde og ingen netværksadgang. Det kræver stadig én ting ud over kvitteringen: udstederens offentlige nøgle, som opnås gennem en kanal, som revisor allerede stoler på, såsom et JWK Set, som udstederen offentliggør, eller en værdi, der er fastlåst på forhånd. Feltet `public_key` inde i kvitteringen kan ikke spille den rolle, fordi den, der skriver kvitteringen, også skriver dette felt. Afsnit 2 viser den forfalskning, dette forhindrer.


In [7]:
def verify_receipt(receipt: dict, trusted_public_key: str) -> bool:
    """
    Verify a receipt's Ed25519 signature against a key you already trust.

    trusted_public_key is the issuer's key, obtained from somewhere other than
    this receipt: a JWK Set, a pinned config value, a key registry. Reading the
    key out of receipt["signature"]["public_key"] instead would only prove the
    receipt is consistent with itself. Section 2 shows exactly how that fails.

    It fails closed: anything malformed returns False instead of raising, so
    one bad receipt cannot stop a batch of them from being checked.
    """
    if not isinstance(receipt, dict) or not isinstance(trusted_public_key, str):
        return False
    sig_obj = receipt.get("signature")
    if not isinstance(sig_obj, dict) or sig_obj.get("alg") != "EdDSA":
        return False
    sig_b64 = sig_obj.get("sig")
    if not isinstance(sig_b64, str):
        return False

    try:
        # Reconstruct the payload that was actually signed (everything except signature).
        payload = {k: v for k, v in receipt.items() if k != "signature"}
        canonical = canonicalize(payload)
        verify_key = signing.VerifyKey(b64url_decode(trusted_public_key))
        verify_key.verify(canonical, b64url_decode(sig_b64))
        return True
    except CryptoError:
        return False
    except Exception:
        # An undecodable key or signature, or a payload that cannot be
        # canonicalized: fail closed rather than raise.
        return False

# public_key_b64 came from Section 1, alongside the signing key. In a real
# deployment it would come from the issuer's JWK Set, not from this notebook.
is_valid = verify_receipt(receipt, public_key_b64)
print(f"Receipt is valid: {is_valid}")

# Regression control for the signature scope in draft revision 02. A receipt
# signed over SHA-256(JCS(payload)) is a signature over different bytes and
# must not verify as a direct-JCS Ed25519 receipt.
prehashed_signature = signing_key.sign(hashlib.sha256(canonicalize(payload)).digest()).signature
prehashed_receipt = {
    **receipt,
    "signature": {**receipt["signature"], "sig": b64url_nopad(prehashed_signature)},
}
print(f"Pre-hashed receipt valid: {verify_receipt(prehashed_receipt, public_key_b64)}")

# Malformed input is rejected, not raised.
for malformed in ({"signature": "x"}, {"signature": {"alg": "EdDSA"}}, "not a receipt"):
    assert verify_receipt(malformed, public_key_b64) is False
print("Malformed receipts rejected: True")

Receipt is valid: True
Pre-hashed receipt valid: False
Malformed receipts rejected: True


Du bør se `Receipt is valid: True`, `Pre-hashed receipt valid: False` og `Malformed receipts rejected: True`. Den positive test beviser, at den direkte-JCS underskrivningsvej virker; den negative kontrol gør signatur-omfangsreglen eksekverbar i stedet for at lade den stå som prosa; den sidste linje viser ugyldigt input, der returnerer `False` i stedet for at rejse en undtagelse.


## Sektion 2: Manipuler kvitteringen

Hele pointen med kvitteringer er, at de er manipulationssikre. Lad os bevise det.

Vi vil ændre præcis ét tegn i kvitteringen og se verifikationen fejle.


In [8]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered, public_key_b64)}")

Original policy_id:  contoso-travel-policy-v3
Tampered policy_id:  contoso-travel-policy-PERMISSIVE

Tampered receipt valid? False


### Hvad skete der lige?

Da vi ændrede `policy_id`, ændrede de kanoniske bytes sig. Signaturen (som var over de oprindelige kanoniske bytes) matcher ikke længere. Verifikationen returnerer korrekt `False`.

Der er ingen måde at ændre et underskrevet felt, altså noget udenfor `signature` objektet, og stadig få kvitteringen til at verificere mod udstederens offentlige nøgle, medmindre angriberen har udstederens private nøgle. Selve `signature` objektet er ikke underskrevet: `sig` er signaturen, og `verify_receipt` ignorerer `public_key`, så blot at ændre det felt ændrer ikke noget. Så længe den private nøgle er i et nøglelager, og verificerende parter bruger udstederens uafhængigt offentliggjorte nøgle, er en ændring i et underskrevet felt umulig at skjule. Den betingelse betyder noget: den næste celle viser en ændret kvittering, der verificeres perfekt mod en anden nøgle.

Prøv det selv: ændr `tool_name` eller `agent_id` eller `timestamp` i cellen ovenfor og kør igen. Hver ændring i et underskrevet felt giver en ugyldig kvittering.


### Manipulationen som signaturen ikke opdager

Angriberen ovenfor underskrev ikke igen. En mere omhyggelig en ville: ændre kvitteringen,
underskrive den med en nøgle, de selv har genereret, og sætte den nøgle i
`public_key`-feltet. Resultatet er en kvittering, hvis signatur er helt ægte
for dens indhold.

Kør dette og se, hvilken af de to kontroller der opdager det.


In [9]:
# The attacker generates their own keypair. Nothing stops them.
attacker_key = signing.SigningKey.generate()

forged_payload = {k: v for k, v in receipt.items() if k != "signature"}
forged_payload["policy_id"] = "contoso-travel-policy-PERMISSIVE"

# Re-signed, with the attacker's key embedded where the issuer's used to be.
forged = sign_receipt(forged_payload, attacker_key, attacker_key.verify_key)

print(f"Issuer's public key : {public_key_b64}")
print(f"Key inside the forgery: {forged['signature']['public_key']}")
print()
print(f"Checked against the key inside it: "
      f"{verify_receipt(forged, forged['signature']['public_key'])}")
print(f"Checked against the issuer's key : "
      f"{verify_receipt(forged, public_key_b64)}")

# Assert the results, so the cell fails if the forgery ever passes as the issuer's.
assert verify_receipt(receipt, public_key_b64) is True                    # genuine, issuer's key
assert verify_receipt(forged, forged["signature"]["public_key"]) is True  # forgery, its own key
assert verify_receipt(forged, public_key_b64) is False                    # forgery, issuer's key

Issuer's public key : NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM
Key inside the forgery: nIXlWGm0syJQysGkn-nRlzFS59HlPCTqeQr32TJoHM4

Checked against the key inside it: True
Checked against the issuer's key : False


Du bør se `True` og derefter `False`. De tre `assert` linjer gør det til en
kontrol: cellen kaster en fejl, hvis forfalskningen nogensinde verificeres mod
udstederens nøgle.

Forfalskningen er internt konsistent. Dens signatur matcher virkelig dens
indhold og den nøgle, den bærer, så en verificerende, der læser nøglen ud af
kvitteringen, rapporterer den som gyldig. Intet ved kvitteringen virker forkert, fordi
intet ved den er inkonsistent.

Det eneste, der fanger den, er at tjekke mod en nøgle, som angriberen ikke havde
mulighed for at påvirke. Derfor tager `verify_receipt` nøglen som et argument: en
funktion med signaturen `verify_receipt(receipt)` kan ikke skrives sikkert,
fordi den eneste nøgle, den kunne få adgang til, er den, en angriber har leveret.

Den samme kvittering gemmes som `sample_receipts/04_resigned_receipt.json`, hvis du
vil inspicere den uden for notebook'en.


## Sektion 3: Kæd kvitteringer sammen

En enkelt kvittering beskytter én handling. De fleste agenter udfører mange handlinger. For at gøre hele sekvensen manipulationssikker, kæder vi hver kvittering til den forrige ved at inkludere den forrige kvitterings hash i den nye kvitterings data.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

Hvis nogen fjerner eller omrokerer en kvittering i den præsenterede sekvens, brydes det næste led. At opdage afkortning eller omskrivning af suffikset styret af underskriveren kræver også et fastgjort aktuelt hoved, et eksternt kontrolpunkt eller en autentificeret befolkningstilsagn.


In [10]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [11]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

Receipt 0: tool=lookup_flights, prev=None
Receipt 1: tool=hold_seat, prev=sha256:af8b9159b25728881ce6a7166429f4451b9fa869a337453d52228f12131192e7
Receipt 2: tool=confirm_booking, prev=sha256:b1634969232be232f53283251cc9307b443a296909d9e74c2b0101b7f4dc679e


In [12]:
def verify_chain(chain: list, trusted_public_key: str) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt must carry previous_receipt_hash: null for the genesis, and
         the previous receipt's hash for every later receipt.
      3. Each sequence number must be an integer equal to the receipt's zero-based
         position in the chain.
    Returns a list of per-receipt result dicts. A malformed entry fails its own
    checks; it does not stop the rest of the chain from being checked.
    """
    results = []
    for i, receipt in enumerate(chain):
        is_receipt = isinstance(receipt, dict)
        sig_ok = verify_receipt(receipt, trusted_public_key)

        # The link field must be present: .get() alone would read a missing field as None.
        has_link = is_receipt and "previous_receipt_hash" in receipt
        link = receipt.get("previous_receipt_hash") if is_receipt else None
        if i == 0:
            chain_ok = has_link and link is None
        else:
            try:
                chain_ok = has_link and link == receipt_hash(chain[i - 1])
            except Exception:
                chain_ok = False  # the previous entry could not be hashed

        # type() rather than isinstance(): bool is a subclass of int, and True == 1.
        sequence = receipt.get("sequence") if is_receipt else None
        seq_ok = type(sequence) is int and sequence == i
        shape_ok = is_receipt and isinstance(receipt.get("tool_name"), str)

        results.append({
            "index": i,
            "tool": str(receipt.get("tool_name", "?")) if is_receipt else "(not a receipt)",
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok and shape_ok,
        })
    return results

for r in verify_chain(chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): VALID
Receipt 2 (   confirm_booking): VALID


Bryd nu kæden ved at manipulere med kvitteringen i midten og verificer igen. Den manipulerede kvittering fejler sin signaturkontrol, OG den næste kvittering fejler sin kædelinkkontrol (fordi dens `previous_receipt_hash` ikke længere matcher den ændrede midterste kvitterings hash).


In [13]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): INVALID (failed: signature)
Receipt 2 (   confirm_booking): INVALID (failed: chain link)


Kvittering 0 verifierer stadig (den blev ikke ændret og har ingen forgænger at afhænge af). Kvittering 1 fejler sin signaturkontrol, fordi vi ændrede `tool_args_hash`. Kvittering 2 fejler sin kædelink-kontrol, fordi dens `previous_receipt_hash` blev beregnet ud fra den originale (nu ændrede) kvittering 1.

Selv hvis en angriber genunderskriver den ændrede kvittering 1, fejler den nye signatur mod udsteders nøgle, medmindre de har udsteders private nøgle (en genunderskrivelse med deres egen nøgle fejler, som Sektion 2 viste), og kædelink-mismatchen i kvittering 2 vil stadig afsløre forfalskningen. For at skjule ændringen skal angriberen genunderskrive alle kvitteringer fra ændringspunktet og frem, hvilket kræver besiddelse af udsteders private nøgle.


## Sektion 4: Pak et agentværktøjskald ind med kvitteringssignering

I en rigtig implementering ønsker du ikke, at hver agentforfatter skal huske at kalde `make_receipt`. Du vil have, at kvitteringssignering sker automatisk for hvert værktøjskald.

Her er det simpleste mønster: en wrapper-klasse, der tager enhver kaldbar værktøjsfunktion og returnerer en kvitteringsudstedende version af den. Dette tilpasser sig enhver agentramme, inklusive Microsoft Agent Framework (`agent_framework.foundry`).

Hvis du ikke har et Microsoft Foundry-projekt sat op, demonstrerer den lokale mock nedenfor stadig mønsteret.


In [14]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [15]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


Tool was called 3 times.
Each call produced a signed receipt linked to the previous one.

Receipt 0 (lookup_flights): VALID
Receipt 1 (lookup_flights): VALID
Receipt 2 (lookup_flights): VALID


### Integration med Microsoft Agent Framework

Wrappet `ReceiptedTool` ovenfor er frameworks-uafhængigt. For at bruge det inde i en agent bygget med Microsoft Agent Framework, skal du registrere den wrapped funktion som et værktøj. Et eksempel (du ville erstatte mocken med en rigtig Microsoft Foundry værktøjsregistrering):

```python
# Pseudocode showing the integration shape.
# import os
# from agent_framework.foundry import FoundryChatClient
# from azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="You are a Contoso Travel agent ...",
#     tools=[receipted_lookup],   # the wrapped tool, not the raw function
# )
# response = agent.run("Find flights from Sydney to Los Angeles in June.")
#
# # After the run, every tool call the agent made has a signed receipt:
# audit_chain = receipted_lookup.receipts
```

Agent-frameworket behøver ikke at kende noget til kvitteringer. Kvitteringssignering er wrapped omkring værktøjet, ikke monteret i frameworket. Sådan tilføjer du provenance til eksisterende agentkode uden at omskrive agenten.


## Opsummering og udfordring

Du har:

- Genereret et Ed25519 nøglepar.
- Oprettet og underskrevet en kvittering for et agentværktøjskald.
- Verificeret kvitteringen offline ved kun at bruge udstederens offentlige nøgle, opnået uafhængigt af kvitteringen.
- Manipuleret med en kvittering og observeret verifikationsfejl.
- Oprettet en hash-kædet sekvens af tre kvitteringer.
- Manipuleret med midten af kæden og observeret både signaturfejl og kædelinksfejl.
- Indkapslet en værktøjsfunktion med automatisk kvitteringsunderskrift.

**Udvidelsesudfordring.** Udvid kvitteringsskemaet med et `request_id` felt (en UUID til distribueret sporing). Opdater `make_receipt` til at inkludere det, og bekræft at kvitteringer fortsat kan verificeres end-to-end. Ændr derefter feltet efter underskrift og bekræft at verificeringen fejler. Dette tvinger dig til at internalisere, hvordan hver byte af den kanoniske kodning bidrager til signaturen.

**Vigtig afgrænsning.** Under en fastlåst nøgle- og kæde-kontekst beviser kvitteringer signaturautorisation, indholdsintegritet og et link til den opgivne forgænger. De beviser IKKE udførelse, kilde-sandhed, betroet tid, fuldstændighed, politikvurdering eller korrekthed. Kvitteringer er en grundpille. Styring er det system, du bygger ovenpå.

Læs lektionens README igen med denne afgrænsning i tankerne. Den mest almindelige fejl, teams begår med kvitteringer, er at antage, at ”vi har kvitteringer” betyder ”vi er styret.” Det gør det ikke. Kvitteringer gør agentadfærd reviderbar. De gør den ikke korrekt.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokument er blevet oversat ved hjælp af AI-oversættelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selvom vi bestræber os på nøjagtighed, skal du være opmærksom på, at automatiserede oversættelser kan indeholde fejl eller unøjagtigheder. Det originale dokument på dets oprindelige sprog bør betragtes som den autoritative kilde. For kritisk information anbefales professionel menneskelig oversættelse. Vi påtager os intet ansvar for misforståelser eller fejltolkninger, der opstår som følge af brugen af denne oversættelse.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
